In [ ]:
import os 
os.environ['AWS_PROFILE'] = 'admin'
os.environ['HAVEN_DATABASE'] = 'haven'

import plotly.express as px
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm import tqdm
from random import choice
import h3

from mirrorverse.utils import read_data_w_cache

In [ ]:
sql = '''
select
    _selected,
    _train,
    _individual,
    _decision,
    _choice,
    depth_bin,
    n_depth_bin,
    h3_index
from
    chinook_depth_inference_3_5_1
where 
    run_id = 'a43c72eff441211221256194305afd6a9ceb5a277a01f3853cf2716fa06725e8'
    and extract(month from time) = 2
'''
data = read_data_w_cache(sql)
data['lat'] = data['h3_index'].apply(lambda h: h3.h3_to_geo(h)[0])
data['lon'] = data['h3_index'].apply(lambda h: h3.h3_to_geo(h)[1])
print(data.shape)
data.head()

In [ ]:
round(data[data['_selected'] & (data['lon'] < -154)].groupby(['depth_bin']).size() / data[data['_selected'] & (data['lon'] < -154)].shape[0], 2)

In [ ]:
round(data[data['_selected'] & (data['lon'] < -154) & data['_train']].groupby(['depth_bin']).size() / data[data['_selected'] & (data['lon'] < -154) & data['_train']].shape[0], 2)

In [ ]:
round(data[data['_selected'] & (data['lon'] > -154) & (data['lon'] < -140) & data['_train']].groupby(['depth_bin']).size() / data[data['_selected'] & (data['lon'] > -154) & (data['lon'] < -140) & data['_train']].shape[0], 2)

In [ ]:
round(data[data['_selected'] & (data['lon'] < -154) & ~data['_train']].groupby(['depth_bin']).size() / data[data['_selected'] & (data['lon'] < -154) & ~data['_train']].shape[0], 2)

In [ ]:
round(data[data['_selected'] & (data['lon'] > -154)].groupby(['depth_bin']).size() / data[data['_selected'] & (data['lon'] > -154)].shape[0], 2)

In [ ]:
data[data['_selected'] & (data['lon'] > -154)].groupby('depth_bin').size()

In [ ]:
data['lon'].min()

In [ ]:
sql = '''
select 
    extract(month from time) as month,
    avg(ln(probability)) as env_model
from
    chinook_depth_inference_3_5_1
where 
    run_id = 'a43c72eff441211221256194305afd6a9ceb5a277a01f3853cf2716fa06725e8'
    and _selected
    and not _train
group by 1 
order by 1
'''
env = read_data_w_cache(sql)
env.head()

In [ ]:
sql = '''
select 
    extract(month from time) as month,
    avg(ln(probability)) as env_model
from
    chinook_depth_inference_3_6_1
where 
    run_id = '1bdad75bbb8cda4a2202db148d44da990e22c4c8cbd84acccfe9e3af75b0a626'
    and _selected
    and not _train
group by 1 
order by 1
'''
env = read_data_w_cache(sql)
env.head()

In [ ]:
sql = '''
select 
    extract(month from time) as month,
    avg(ln(probability)) as env_model
from
    chinook_depth_inference_3_7_2
where 
    run_id = '4833fde63cca7a92d04cc788b6d0273871771546e8d02a833edfdc6897b4af96'
    and _selected
    and not _train
group by 1 
order by 1
'''
env = read_data_w_cache(sql)
env.head()

In [ ]:
sql = '''
select 
    extract(month from time) as month,
    avg(ln(probability)) as base_model
from
    chinook_depth_inference_3_1_4
where 
    run_id = '00cf23b296999368ea18b82e33b8687c51e8c35e876afd325e26317cb69ea45b'
    and _selected
    and not _train
group by 1 
order by 1
'''
base = read_data_w_cache(sql)
base.head()

In [ ]:
df = env.merge(base)
df['diff'] = df['env_model'] - df['base_model']
df

In [ ]:
df = env.merge(base)
df['diff'] = df['env_model'] - df['base_model']
df

In [ ]:
model = '3_5_1'
run_id = 'a43c72eff441211221256194305afd6a9ceb5a277a01f3853cf2716fa06725e8'

data = read_data_w_cache(
    f"""
        with data as (
            select 
                *, 
                month(time) as month, 
                max(depth_bin) over (partition by h3_index) as max_depth_bin 
            from 
                chinook_depth_full_inference_{model} 
            where 
                run_id = '{run_id}'
        )
        select * from data where depth_bin = 25 and max_depth_bin = 150
    """
)
data['radians'] = np.arctan2(data['sin_sun'], data['cos_sun'])
print(data.shape)
data.head()

In [ ]:
data['binned_radians'] = round(data['radians'] / (np.pi / 8)) * (np.pi / 8)
data['binned_habitat'] = round(data['habitat'] / (0.1)) * (0.1)
df = data[(data['depth_bin'] == 25)]
gdf = df.groupby(['month', 'binned_habitat'])[['probability']].mean().reset_index()
px.scatter(
    gdf, x='month', y='probability', color='binned_habitat'
)

In [ ]:
model = '3_5_1' #'3_1_18'
run_id = 'a43c72eff441211221256194305afd6a9ceb5a277a01f3853cf2716fa06725e8' #'dcdcf74abea8d96ff28901fcd9653fc783df9cdda20ed550adc8c2c38a11d896'
data = read_data_w_cache(
    f"""
        with data as (
            select 
                *, 
                month(time) as month, 
                max(depth_bin) over (partition by h3_index) as max_depth_bin 
            from 
                chinook_depth_inference_{model} 
            where 
                run_id = '{run_id}'
        )
        select * from data where depth_bin = 25
    """
)
data['radians'] = np.arctan2(data['sin_sun'], data['cos_sun'])
data['binned_radians'] = round(data['radians'] / (np.pi / 8)) * (np.pi / 8)
data['binned_habitat'] = round(data['habitat'] / (0.1)) * (0.1)
print(data.shape)
data.head()

In [ ]:
px.scatter(
    data[data['month'] == 1].groupby(['max_depth_bin', 'binned_habitat']).size().reset_index(), x='max_depth_bin', y='binned_habitat', size=0
)

In [ ]:
sample_size = 100
max_depth_bins = set([150, 200])

dfs = []
for month in tqdm(sorted(data['month'].unique())):
    sdf = data[data['month'] == month]
    if max_depth_bins - set(sdf['max_depth_bin'].unique()):
        continue
    for max_depth_bin in max_depth_bins:
        dfs.append(
            sdf[sdf['max_depth_bin'] == max_depth_bin].sample(sample_size, replace=True)
        )
df = pd.concat(dfs)

In [ ]:
gdf = df[df['month'] == 10].groupby(['binned_habitat']).agg({'_selected': 'mean', '_individual': 'nunique'}).reset_index()
gdf = gdf[gdf['_individual'] >= 5]
px.scatter(
    gdf, x='binned_habitat', y='_selected',
    height=333, size='_individual'
)

In [ ]:
# (4, 500), (7, 500), (10, 100/150), (11, 500), ~(1, 500), (3, 200), (9, 100-200)
# (7, 200/250), (8, 100/150), (5, 150/200)
px.scatter(
    data[data['month'] == 9].groupby(['max_depth_bin', 'binned_habitat'])['_individual'].nunique().reset_index(), x='max_depth_bin', y='binned_habitat', size='_individual'
)

In [ ]:
sample_size = 1000
max_depth_bins = set([100, 150, 200])

dfs = []
for month in tqdm(sorted(data['month'].unique())):
    sdf = data[data['month'] == month]
    if max_depth_bins - set(sdf['max_depth_bin'].unique()):
        continue
    for max_depth_bin in max_depth_bins:
        dfs.append(
            sdf[sdf['max_depth_bin'] == max_depth_bin].sample(sample_size, replace=True)
        )
df = pd.concat(dfs)

In [ ]:
gdf = data[data['month'] == 9].groupby(['binned_habitat']).agg({'_selected': 'mean', '_individual': 'nunique'}).reset_index()
gdf = gdf[gdf['_individual'] >= 5]
px.scatter(
    gdf, x='binned_habitat', y='_selected',
    height=333, size='_individual'
)

In [ ]:
gdf = df[df['month'] == 2].groupby(['binned_habitat']).agg({'probability': 'mean', '_individual': 'nunique'}).reset_index()
#gdf = gdf[gdf['_individual'] >= 5]
px.scatter(
    gdf, x='binned_habitat', y='probability',
    height=333, size='_individual'
)

In [ ]:
px.scatter(
    data[data['month'] == 8].groupby(['max_depth_bin', 'binned_habitat']).size().reset_index(), x='max_depth_bin', y='binned_habitat', size=0
)

In [ ]:
sample_size = 100
max_depth_bins = set([100, 150, 200])

dfs = []
for month in tqdm(sorted(data['month'].unique())):
    sdf = data[data['month'] == month]
    if max_depth_bins - set(sdf['max_depth_bin'].unique()):
        continue
    for max_depth_bin in max_depth_bins:
        dfs.append(
            sdf[sdf['max_depth_bin'] == max_depth_bin].sample(sample_size, replace=True)
        )
df = pd.concat(dfs)

In [ ]:
gdf = df[df['month'] == 8].groupby(['binned_habitat']).agg({'_selected': 'mean', '_individual': 'nunique'}).reset_index()
gdf = gdf[gdf['_individual'] >= 5]
px.scatter(
    gdf, x='binned_habitat', y='_selected',
    height=333, size='_individual'
)

In [ ]:
px.scatter(
    df[df['month'] == 10].groupby(['binned_habitat']).agg({'_selected': 'mean', '_individual': 'nunique'}).reset_index(), x='binned_habitat', y='_selected',
    height=333, size='_individual'
)

In [ ]:
data['binned_radians'] = round(data['radians'] / (np.pi / 8)) * (np.pi / 8)
data['binned_habitat'] = round(data['habitat'] / (0.1)) * (0.1)
df = data[(data['depth_bin'] == 25)]
gdf = df.groupby(['binned_radians', 'month', 'binned_habitat']).agg({'_selected': 'mean', '_individual': 'nunique'}).reset_index()
px.scatter(
    gdf, x='binned_radians', y='_selected', facet_col='month', color='binned_habitat', facet_col_wrap=4,
    category_orders={"month": range(1, 13), "binned_habitat": sorted(gdf['binned_habitat'].unique())},
    height=1000, size='_individual'
)

In [ ]:
data['binned_radians'] = round(data['radians'] / (np.pi / 8)) * (np.pi / 8)
data['binned_habitat'] = round(data['habitat'] / (0.1)) * (0.1)
df = data[(data['depth_bin'] == 25)]
gdf = df.groupby(['binned_radians', 'month', 'binned_habitat'])[['probability']].mean().reset_index()
px.scatter(
    gdf, x='binned_radians', y='probability', facet_col='month', color='binned_habitat', facet_col_wrap=4,
    category_orders={"month": range(1, 13), "binned_habitat": sorted(gdf['binned_habitat'].unique())},
    height=1000
)

In [ ]:
data['binned_radians'] = round(data['radians'] / (np.pi / 8)) * (np.pi / 8)
data['binned_habitat'] = round(data['habitat'] / (0.1)) * (0.1)
df = data[(data['depth_bin'] == 25)]
gdf = df.groupby(['binned_radians', 'month', 'binned_habitat']).agg({'_selected': 'mean', '_individual': 'nunique'}).reset_index()
gdf = gdf[gdf['_individual'] > 5]
px.scatter(
    gdf, x='binned_radians', y='_selected', facet_col='month', color='binned_habitat', facet_col_wrap=4,
    category_orders={"month": range(1, 13), "binned_habitat": sorted(gdf['binned_habitat'].unique())},
    height=1000, size='_individual'
)

In [ ]:
df = data[(data['depth_bin'] == 25) & (data['elevation'] < -50) & (data['elevation'] > -150)]
gdf = df.groupby(['binned_radians', 'month', 'binned_habitat'])[['probability']].mean().reset_index()
px.scatter(
    gdf, x='binned_radians', y='probability', facet_col='month', color='binned_habitat', facet_col_wrap=4,
    category_orders={"month": range(1, 13), "binned_habitat": sorted(gdf['binned_habitat'].unique())},
    height=1000
)

In [ ]:
df = data[(data['depth_bin'] == 25) & (data['elevation'] < -50) & (data['elevation'] > -150)]
gdf = df.groupby(['binned_radians', 'month', 'binned_habitat']).agg({'_selected': 'mean', '_individual': 'nunique'}).reset_index()
px.scatter(
    gdf, x='binned_radians', y='_selected', facet_col='month', color='binned_habitat', facet_col_wrap=4,
    category_orders={"month": range(1, 13), "binned_habitat": sorted(gdf['binned_habitat'].unique())},
    height=1000, size='_individual'
)

In [ ]:
df = data[(data['depth_bin'] == 25)]
gdf = df.groupby(['month', 'binned_habitat']).agg({'_selected': 'mean', '_individual': 'nunique'}).reset_index()
px.scatter(
    gdf, x='month', y='_selected', size='_individual', color='binned_habitat', facet_col_wrap=4,
    height=1000
)

In [ ]:
data['binned_radians'] = round(data['radians'] / (np.pi/8)) * (np.pi/8)
df = data[(data['depth_bin'] == 25)]
gdf = df.groupby(['binned_radians', 'month'])[['_selected']].mean().reset_index()
px.scatter(
    gdf, x='binned_radians', y='_selected', facet_col='month', facet_col_wrap=3,
    category_orders={"month": range(1, 13)},
    height=1500
)

In [ ]:
data[data['depth_bin'] == 25]['_selected'].mean()

In [ ]:
data[(data['depth_bin'] == 25) & (data['depth_bin'])]['probability'].mean()